# 05 — Password and Credential Security Analysis

## What This Is
Credential analysis covers password strength evaluation, hash identification, common password patterns, and credential stuffing detection. Defenders use these techniques to enforce strong password policies and detect breached credentials.

## Why It Matters
The 2023 Verizon DBIR found credentials are the #1 attack vector — involved in 86% of web application breaches. Understanding how passwords fail is essential for building authentication systems that resist attacks.

## Where the Community Stands
HIBP (Have I Been Pwned) provides breach corpus lookup. NIST SP 800-63B (2017) overhauled password guidance: ban common passwords, allow long passphrases, drop mandatory rotation. Modern systems check passwords against breach lists at registration.

## Authorized Testing Context
All analysis uses synthetic credentials and publicly-documented patterns. Never attempt to crack hashes from systems you are not authorised to test.

In [ ]:
import hashlib
import hmac
import re
import math
import secrets
import string
from dataclasses import dataclass
from typing import List, Tuple, Dict

# Common weak passwords (subset of widely-known lists like rockyou)
COMMON_PASSWORDS = [
    'password', '123456', 'password123', 'qwerty', 'letmein',
    'welcome', 'monkey', 'dragon', '111111', 'baseball',
    'iloveyou', 'master', 'sunshine', 'princess', 'shadow',
    'admin', 'root', 'pass', 'test', 'guest',
]
COMMON_SET = set(COMMON_PASSWORDS)

# Leet-speak substitutions
LEET = {'a': '@', 'e': '3', 'i': '1', 'o': '0', 's': '$', 't': '7'}

def entropy_bits(password: str) -> float:
    """Shannon entropy in bits."""
    if not password:
        return 0.0
    freq = {}
    for c in password:
        freq[c] = freq.get(c, 0) + 1
    n = len(password)
    return -sum((f/n)*math.log2(f/n) for f in freq.values())

def charset_size(password: str) -> int:
    size = 0
    if any(c.islower() for c in password): size += 26
    if any(c.isupper() for c in password): size += 26
    if any(c.isdigit() for c in password): size += 10
    if any(not c.isalnum() for c in password): size += 32
    return size

def guesses_to_crack(password: str) -> float:
    """Approximate guesses using charset^length model."""
    cs = charset_size(password)
    return cs ** len(password) if cs > 0 else 1.0

print('Password analysis utilities defined')

In [ ]:
@dataclass
class PasswordStrength:
    password: str
    length: int
    entropy: float
    charset: int
    is_common: bool
    has_leet: bool
    has_keyboard_walk: bool
    score: int          # 0-100
    label: str

KEYBOARD_WALKS = ['qwerty', 'asdf', 'zxcv', '1234', '2345', 'qazwsx', 'wsxedc']

def analyse_password(pwd: str) -> PasswordStrength:
    lower = pwd.lower()
    # De-leet to check if underlying word is common
    de_leeted = lower
    for leet_char, orig in [('@','a'),('3','e'),('1','i'),('0','o'),('$','s'),('7','t')]:
        de_leeted = de_leeted.replace(leet_char, orig)

    is_common      = lower in COMMON_SET or de_leeted in COMMON_SET
    has_leet       = de_leeted != lower and de_leeted in COMMON_SET
    has_walk       = any(walk in lower for walk in KEYBOARD_WALKS)
    ent            = entropy_bits(pwd)
    cs             = charset_size(pwd)

    # Scoring (0-100)
    score = 0
    score += min(40, len(pwd) * 3)         # length: up to 40
    score += min(20, cs // 2)              # charset: up to 20 (cs max ~94)
    score += min(20, int(ent * 3))         # entropy: up to 20
    if is_common:  score -= 40
    if has_walk:   score -= 15
    if has_leet:   score -= 10  # leet helps less than it seems
    score = max(0, min(100, score))

    if score >= 70:   label = 'STRONG'
    elif score >= 45: label = 'MODERATE'
    elif score >= 20: label = 'WEAK'
    else:             label = 'VERY WEAK'

    return PasswordStrength(
        password=pwd, length=len(pwd), entropy=round(ent,2),
        charset=cs, is_common=is_common, has_leet=has_leet,
        has_keyboard_walk=has_walk, score=score, label=label
    )

test_passwords = [
    'password', 'P@ssw0rd', 'qwerty123!', 'correct-horse-battery-staple',
    'Tr0ub4dor&3', 'hunter2', 'xK9#mL2$vN8@qR5!', 'abc123'
]

print(f'{"Password":<35} {"Score":<8} {"Label":<12} {"Common":<8} {"Entropy"}')
print('-' * 80)
for p in test_passwords:
    s = analyse_password(p)
    print(f'{p:<35} {s.score:<8} {s.label:<12} {str(s.is_common):<8} {s.entropy:.2f}')

## Hash Identification and Secure Storage
Password hashes vary dramatically in security. MD5/SHA1 are broken for passwords (GPU can crack billions/sec). bcrypt/scrypt/Argon2 are purpose-built — deliberately slow with tunable cost.

In [ ]:
def identify_hash(hash_str: str) -> str:
    """Heuristic hash type identification."""
    h = hash_str.strip()
    if h.startswith('$2b$') or h.startswith('$2a$'):
        return 'bcrypt'
    if h.startswith('$argon2'):
        return 'Argon2'
    if h.startswith('$6$'):
        return 'SHA-512 crypt'
    if h.startswith('$5$'):
        return 'SHA-256 crypt'
    if h.startswith('$1$'):
        return 'MD5 crypt (insecure)'
    if re.match(r'^[0-9a-fA-F]{32}$', h):
        return 'MD5 (likely, 32 hex — insecure)'
    if re.match(r'^[0-9a-fA-F]{40}$', h):
        return 'SHA-1 (40 hex — insecure)'
    if re.match(r'^[0-9a-fA-F]{64}$', h):
        return 'SHA-256 (64 hex — insecure without KDF)'
    if re.match(r'^[0-9a-fA-F]{128}$', h):
        return 'SHA-512 (128 hex — insecure without KDF)'
    return 'Unknown'

# Simulate secure vs insecure storage
test_hashes = [
    ('MD5 raw',      hashlib.md5(b'password').hexdigest()),
    ('SHA1 raw',     hashlib.sha1(b'password').hexdigest()),
    ('SHA256 raw',   hashlib.sha256(b'password').hexdigest()),
    ('bcrypt-like',  '$2b$12$LQv3c1yqBWVHxkd0LHAkCOYz6TtxMQJqhN8/LewJWYk1234567890ab'),
    ('Argon2-like',  '$argon2id$v=19$m=65536,t=3,p=4$c2FsdHNhbHRz$hash'),
]

print(f'{"Algorithm":<15} {"Identified":<35} {"Secure?"}')
print('-' * 65)
for name, h in test_hashes:
    ident   = identify_hash(h)
    secure  = 'YES' if any(x in ident for x in ['bcrypt','Argon2','SHA-512 crypt','SHA-256 crypt']) else 'NO'
    print(f'{name:<15} {ident:<35} {secure}')

## Credential Stuffing Detection
Credential stuffing attacks reuse leaked username/password pairs across sites. Defenders detect them via rate limiting, geographic anomalies, and login velocity monitoring.

In [ ]:
import collections

@dataclass
class LoginEvent:
    timestamp: float
    ip: str
    username: str
    success: bool

class CredentialStuffingDetector:
    def __init__(self, ip_threshold=10, user_threshold=5, window_sec=300):
        self.ip_threshold   = ip_threshold
        self.user_threshold = user_threshold
        self.window_sec     = window_sec
        self.events: List[LoginEvent] = []

    def add_event(self, event: LoginEvent):
        self.events.append(event)

    def analyse(self, as_of: float) -> Dict:
        window_start = as_of - self.window_sec
        recent = [e for e in self.events if e.timestamp >= window_start]

        ip_counts   = collections.Counter(e.ip for e in recent if not e.success)
        user_counts = collections.Counter(e.username for e in recent if not e.success)

        suspicious_ips   = {ip: cnt for ip, cnt in ip_counts.items()  if cnt >= self.ip_threshold}
        stuffed_users    = {u:  cnt for u,  cnt in user_counts.items() if cnt >= self.user_threshold}

        total  = len(recent)
        fails  = sum(1 for e in recent if not e.success)
        rate   = fails / total if total > 0 else 0.0

        return {
            'window_events':   total,
            'failures':        fails,
            'failure_rate':    round(rate, 3),
            'suspicious_ips':  suspicious_ips,
            'stuffed_users':   stuffed_users,
            'alert':           bool(suspicious_ips or stuffed_users or rate > 0.5),
        }

# Simulate a credential stuffing attack
import time
NOW = 1000.0
rng = random.Random(42)

detector = CredentialStuffingDetector()
# Normal traffic
for i in range(20):
    detector.add_event(LoginEvent(
        timestamp=NOW - rng.uniform(0, 300),
        ip=f'192.168.{rng.randint(1,10)}.{rng.randint(1,254)}',
        username=f'user{rng.randint(1,50)}',
        success=rng.random() > 0.2
    ))
# Credential stuffing from attacker IP
for i in range(15):
    detector.add_event(LoginEvent(
        timestamp=NOW - rng.uniform(0, 60),
        ip='203.0.113.42',  # attacker
        username=f'victim{i}@example.com',
        success=False
    ))

report = detector.analyse(NOW)
print('=== Credential Stuffing Detection ===')
for k, v in report.items():
    print(f'  {k}: {v}')